# Circular research agent in Colab
Run the model and agent in this notebook's own runtime. No tunnels or web UI. Select a GPU runtime if available. The default documents are fictional and must not be used as regulatory guidance. Model downloads and inference use your runtime quota.


In [ ]:
import sys, subprocess
print(sys.version)
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'Use Python 3.11–3.13'
subprocess.run(['nvidia-smi'], check=False)


## Upload the source ZIP
Use the rbi-sebi-agent.zip produced by scripts/package.py. The notebook extracts it under /content/rbi-sebi-agent.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive, 'Upload the project ZIP'
root = Path('/content/rbi-sebi-agent').resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archive])) as z:
    assert all((root / name).resolve().is_relative_to(root) for name in z.namelist()), 'Unsafe archive path'
    z.extractall(root)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ci.txt'], check=True)


In [ ]:
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True)
subprocess.run(['sudo', 'apt-get', 'install', '-y', 'pciutils', 'zstd'], check=True)
subprocess.run(['curl', '-fsSL', 'https://ollama.com/install.sh', '-o', '/content/install-ollama.sh'], check=True)
subprocess.run(['sh', '/content/install-ollama.sh'], check=True)


In [ ]:
import urllib.request, time
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
log = open('/content/ollama.log', 'w')
server = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
for attempt in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).close()
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start; inspect /content/ollama.log')
os.environ['LLM_MODEL'] = 'qwen3:8b'
os.environ['LLM_TIMEOUT'] = '180'
subprocess.run(['ollama', 'pull', os.environ['LLM_MODEL']], check=True)
subprocess.run([sys.executable, '-m', 'scripts.smoke'], check=True)
subprocess.run(['ollama', 'ps'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'src.retrieval.index', '--chunks', 'tests/fixtures/chunks.jsonl'], check=True)
subprocess.run([sys.executable, '-m', 'src.retrieval.index', '--chunks', 'tests/fixtures/attack_chunks.jsonl', '--out', 'index_attack'], check=True)


# Resumable evaluation
Run the setup cells above first. This example measures fictional dev questions, not official regulatory accuracy. Upload official PDFs and a verified manifest/question set before official evaluation.


In [ ]:
subprocess.run([sys.executable, '-m', 'evals.run', '--questions', 'tests/fixtures/questions.jsonl', '--fixture', '--split', 'dev', '--out', 'results/colab_fixture_dev.json'], check=True)


In [ ]:
# Explicitly run each defense level when ready; these jobs may take a long time.
# subprocess.run([sys.executable, '-m', 'redteam.run', '--fixture', '--defense', '0', '--out', 'results/colab_redteam_d0.json'], check=True)
# subprocess.run([sys.executable, '-m', 'redteam.run', '--fixture', '--defense', '5', '--out', 'results/colab_redteam_d5.json'], check=True)


In [ ]:
import shutil
shutil.make_archive('/content/results-and-traces', 'zip', '.', 'results')
files.download('/content/results-and-traces.zip')
# Also preserve raw traces before ending the runtime.
shutil.make_archive('/content/traces', 'zip', '.', 'runs')
files.download('/content/traces.zip')
